# Conclusiones

In [1]:
import json
import sys
import warnings
from pathlib import Path

RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from src import analisis, datos
from src import estadistica as est
from src.analisis import NOMBRES_MODELO
from src.formato import dec, ent, enumerar, p_valor, pct

tabla = analisis.cargar_tabla()
ok = tabla[tabla['estado'] == 'ok']
guia = ok[ok['modelo'] != 'ebm']
mejores = {t: analisis.mejores_por_modelo(guia, t) for t in ('clasificacion', 'regresion')}
CMP = RAIZ / 'resultados' / 'comparacion_entrega2'
COMP = RAIZ / 'resultados' / 'complementos'
COMPUTO = RAIZ / 'resultados' / 'computo'
busq = json.loads((CMP / 'busquedas.json').read_text(encoding='utf-8'))
c_clf = pd.read_csv(CMP / 'contrastes_clasificacion.csv').set_index(['referencia', 'candidato'])
c_reg = pd.read_csv(CMP / 'contrastes_regresion.csv').set_index(['referencia', 'candidato'])
metricas = pd.read_csv(RAIZ / 'resultados' / 'metricas_prueba.csv') if (RAIZ / 'resultados' / 'metricas_prueba.csv').exists() else None
resumen_opt = analisis.resumen_optimizadores(tabla[tabla['modelo'] != 'ebm'])
_, por_busqueda = analisis.curvas_anytime(tabla[tabla['modelo'] != 'ebm'], 'clasificacion')
fr_area = est.friedman(por_busqueda.pivot_table(index=['modelo', 'balanceo', 'pliegue'], columns='optimizador',
                                                values='area')[analisis.OPTIMIZADORES], mayor_es_mejor=False)
mc, mr = mejores['clasificacion'], mejores['regresion']
techo = json.loads((CMP / 'techo.json').read_text(encoding='utf-8'))
e_clf, e_reg = c_clf.loc[('logistica_e2', 'ebm')], c_reg.loc[('lasso_e2', 'ebm')]
x_clf, x_reg = c_clf.loc[('xgboost', 'ebm')], c_reg.loc[('xgboost', 'ebm')]

display(Markdown(f"""
**1. El problema real es más difícil de lo que parecía, y la causa está en los datos.** La probabilidad
de fuga de COFINFAD no es un evento observado sino un puntaje generado en buena parte a partir del
número de productos activos (una escalera de paso constante), y por eso esa variable se excluyó como
predictor. Sin ella, la señal se concentra en cuatro variables (frecuencia de uso de la aplicación, edad
—que solo toma tres cohortes—, satisfacción y satisfacción de base), con efectos no lineales, y existe un
techo: ningún modelo puede pasar de un AUC de unas {dec(techo['auc_mediana'], 3)} con la etiqueta de la mediana ni de un R²
de unas {dec(techo['r2'], 3)} con estas variables. Las entregas anteriores atribuían el bajo desempeño a los modelos; el
diagnóstico muestra que una parte era inevitable.

**2. Los modelos de árboles son los mejores y no se distinguen entre sí.** En clasificación,
{enumerar(f"{NOMBRES_MODELO[f.modelo]} ({dec(f.metrica)})" for f in mc.head(3).itertuples())} encabezan la validación anidada, y las pruebas de
Friedman, Nemenyi y DeLong no logran separarlos; k-NN, la regresión logística, el SVM lineal y Naive
Bayes quedan claramente por debajo. En regresión el orden es parecido
({enumerar(f"{NOMBRES_MODELO[f.modelo]} {dec(f.metrica)}" for f in mr.head(3).itertuples())} de RMSE) y el *Model Confidence Set* se queda con
{NOMBRES_MODELO[mr.iloc[0]['modelo']]}, aunque con ventajas de décimas de milésima. Con diferencias de este tamaño, la elección
del modelo para producción debe apoyarse en la calibración, el coste y la interpretabilidad.

**3. El modelo nuevo cumple la exigencia de la entrega.** La EBM supera a la Entrega 2 en la misma
prueba y con la misma etiqueta: {dec(e_clf['diferencia'], 4, True)} de AUC (DeLong {p_valor(e_clf['p_delong'])}) y {dec(e_reg['mejora_r2'], 4, True)} de R²
(Diebold-Mariano con HLN {p_valor(e_reg['p_dm'])}; bootstrap estacionario {p_valor(e_reg['p_bootstrap_estacionario'])}), con intervalos bootstrap
que no incluyen el cero y mejoras que se repiten en todos los pliegues. Empata con XGBoost ({p_valor(x_clf['p_delong'])}
y {p_valor(x_reg['p_dm'])}) y explica sus predicciones sin herramientas auxiliares: sus funciones de forma
muestran exactamente la no linealidad que los modelos lineales de la Entrega 2 no podían captar.

**4. La optimización bayesiana es la más eficiente por evaluación; la métrica final no distingue a los
métodos.** Con el mismo presupuesto, TPE cierra antes la brecha hasta la mejor configuración (Friedman
sobre el área bajo la curva de arrepentimiento, {p_valor(fr_area['p'])}). El genético termina con el mejor rango en
la métrica externa de clasificación, Random queda por debajo de Grid de forma consistente, y Grid es el
menos eficiente. En tiempo de reloj la ventaja se invierte en un equipo de 6 núcleos, porque la
bayesiana y la genética son secuenciales: tardaron {dec(resumen_opt.loc[('clasificacion', 'bayesiana'), 's_por_evaluacion'] / resumen_opt.loc[('clasificacion', 'grid'), 's_por_evaluacion'], 1)} veces más por evaluación.
La validación anidada funcionó: el AUC externo predice el de prueba casi sin sesgo.
"""))


**1. El problema real es más difícil de lo que parecía, y la causa está en los datos.** La probabilidad
de fuga de COFINFAD no es un evento observado sino un puntaje generado en buena parte a partir del
número de productos activos (una escalera de paso constante), y por eso esa variable se excluyó como
predictor. Sin ella, la señal se concentra en cuatro variables (frecuencia de uso de la aplicación, edad
—que solo toma tres cohortes—, satisfacción y satisfacción de base), con efectos no lineales, y existe un
techo: ningún modelo puede pasar de un AUC de unas 0,722 con la etiqueta de la mediana ni de un R²
de unas 0,220 con estas variables. Las entregas anteriores atribuían el bajo desempeño a los modelos; el
diagnóstico muestra que una parte era inevitable.

**2. Los modelos de árboles son los mejores y no se distinguen entre sí.** En clasificación,
XGBoost (0,8332), Árbol de decisión (0,8332) y Random Forest (0,8326) encabezan la validación anidada, y las pruebas de
Friedman, Nemenyi y DeLong no logran separarlos; k-NN, la regresión logística, el SVM lineal y Naive
Bayes quedan claramente por debajo. En regresión el orden es parecido
(XGBoost 0,0590, Random Forest 0,0592 y Árbol de decisión 0,0593 de RMSE) y el *Model Confidence Set* se queda con
XGBoost, aunque con ventajas de décimas de milésima. Con diferencias de este tamaño, la elección
del modelo para producción debe apoyarse en la calibración, el coste y la interpretabilidad.

**3. El modelo nuevo cumple la exigencia de la entrega.** La EBM supera a la Entrega 2 en la misma
prueba y con la misma etiqueta: +0,0355 de AUC (DeLong p < 10⁻¹⁸) y +0,0648 de R²
(Diebold-Mariano con HLN p < 10⁻³⁹; bootstrap estacionario p < 0,001), con intervalos bootstrap
que no incluyen el cero y mejoras que se repiten en todos los pliegues. Empata con XGBoost (p = 0,196
y p = 0,074) y explica sus predicciones sin herramientas auxiliares: sus funciones de forma
muestran exactamente la no linealidad que los modelos lineales de la Entrega 2 no podían captar.

**4. La optimización bayesiana es la más eficiente por evaluación; la métrica final no distingue a los
métodos.** Con el mismo presupuesto, TPE cierra antes la brecha hasta la mejor configuración (Friedman
sobre el área bajo la curva de arrepentimiento, p < 10⁻⁷). El genético termina con el mejor rango en
la métrica externa de clasificación, Random queda por debajo de Grid de forma consistente, y Grid es el
menos eficiente. En tiempo de reloj la ventaja se invierte en un equipo de 6 núcleos, porque la
bayesiana y la genética son secuenciales: tardaron 1,7 veces más por evaluación.
La validación anidada funcionó: el AUC externo predice el de prueba casi sin sesgo.


In [2]:
partes = []
if (COMP / 'calibracion.csv').exists():
    cal = pd.read_csv(COMP / 'calibracion.csv')
    piv = cal.pivot_table(index='modelo', columns='version', values='ece')
    partes.append(f"""**5. El balanceo no mejora la discriminación; mueve el punto de operación, y la calibración se
corrige aparte.** `class_weight` deja el AUC igual y SMOTE y ADASYN lo empeoran en casi todos los
modelos; lo que cambian es la probabilidad media y, con ella, el recall con el umbral de 0,5. Para una
campaña de retención conviene fijar el número de clientes a contactar y recalibrar las probabilidades:
la recalibración isotónica, ajustada con predicciones fuera de pliegue, lleva el ECE de entre
{dec(piv['sin recalibrar'].min(), 3)} y {dec(piv['sin recalibrar'].max(), 3)} a entre {dec(piv['isotónica'].min(), 3)} y {dec(piv['isotónica'].max(), 3)} sin cambiar el orden de los clientes.""")
else:
    partes.append("""**5. El balanceo no mejora la discriminación; mueve el punto de operación.** `class_weight` deja el
AUC igual y SMOTE y ADASYN lo empeoran en casi todos los modelos; lo que cambian es la probabilidad
media y, con ella, el recall con el umbral de 0,5. Para una campaña de retención conviene fijar el
número de clientes a contactar y calibrar las probabilidades.""")
if (COMPUTO / 'xgboost.csv').exists() and (COMPUTO / 'knn.csv').exists():
    xg = pd.read_csv(COMPUTO / 'xgboost.csv')
    m = xg[xg['experimento'] == 'método y dispositivo'].set_index('variante')
    t_exact = m[m.index.str.startswith('exact')]['entrenamiento_s'].iloc[0]
    t_hist = m.loc['hist · cpu · todos los hilos', 'entrenamiento_s']
    t_gpu = m[m.index.str.contains('cuda')]['entrenamiento_s'].iloc[0]
    knn = pd.read_csv(COMPUTO / 'knn.csv')
    k78 = knn[knn['dimension'] == '78 variables'].set_index('metodo')
    k4 = knn[knn['dimension'] == '4 variables con señal'].set_index('metodo')
    gpu = ('la GPU no acelera con este tamaño de datos' if t_gpu >= t_hist else
           f'la GPU acelera {dec(t_hist / t_gpu, 1)} veces')
    partes.append(f"""**6. Las decisiones de cómputo se sostienen con mediciones.** El método `hist` de XGBoost entrena
{dec(t_exact / t_hist, 1)} veces más rápido que el exacto con el mismo AUC, y {gpu}. En k-NN, con 78 variables, ni
KD-Tree ni Ball Tree ni FAISS mejoran a la fuerza bruta de scikit-learn ({dec(k78.loc['scikit-learn brute', 'consulta_s'], 1)} s por consulta de
toda la prueba): lo que compensa es quedarse con las 4 variables con señal, que bajan la consulta a
{dec(k4['consulta_s'].min(), 1)} s y suben el AUC de {dec(k78.loc['scikit-learn brute', 'auc'], 3)} a {dec(k4.loc['scikit-learn brute', 'auc'], 3)}. El SVM con núcleo multiplicaría el coste por
decenas, y la multi-fidelidad de Random Forest y la EBM conserva la configuración ganadora con una fracción
del coste.""")
else:
    partes.append("""**6. Las decisiones de cómputo se sostienen con mediciones** (sección de optimización computacional):
`hist` frente a `exact` en XGBoost, FAISS frente a los árboles de búsqueda en k-NN, LinearSVC frente al
núcleo RBF y la multi-fidelidad de Random Forest y la EBM.""")
partes.append("""**7. Limitaciones y trabajo futuro.** COFINFAD es un conjunto sintético: las cohortes de edad, la
escalera de `active_products` y la ausencia de señal en las transacciones lo delatan, y las
conclusiones sobre qué determina la fuga no deben trasladarse sin más a clientes reales. La etiqueta es
un puntaje de otro modelo, no la fuga observada, de modo que lo que se aprende es a reproducir ese
puntaje. Con datos reales convendría validar en el tiempo (entrenar con un periodo y probar con el
siguiente), incorporar el coste de cada acción de retención para elegir el umbral, y aprovechar las
secuencias de transacciones con modelos que las traten como series.""")
display(Markdown('\n\n'.join(partes)))

**5. El balanceo no mejora la discriminación; mueve el punto de operación, y la calibración se
corrige aparte.** `class_weight` deja el AUC igual y SMOTE y ADASYN lo empeoran en casi todos los
modelos; lo que cambian es la probabilidad media y, con ella, el recall con el umbral de 0,5. Para una
campaña de retención conviene fijar el número de clientes a contactar y recalibrar las probabilidades:
la recalibración isotónica, ajustada con predicciones fuera de pliegue, lleva el ECE de entre
0,027 y 0,682 a entre 0,003 y 0,010 sin cambiar el orden de los clientes.

**6. Las decisiones de cómputo se sostienen con mediciones.** El método `hist` de XGBoost entrena
7,0 veces más rápido que el exacto con el mismo AUC, y la GPU no acelera con este tamaño de datos. En k-NN, con 78 variables, ni
KD-Tree ni Ball Tree ni FAISS mejoran a la fuerza bruta de scikit-learn (8,0 s por consulta de
toda la prueba): lo que compensa es quedarse con las 4 variables con señal, que bajan la consulta a
1,8 s y suben el AUC de 0,787 a 0,821. El SVM con núcleo multiplicaría el coste por
decenas, y la multi-fidelidad de Random Forest y la EBM conserva la configuración ganadora con una fracción
del coste.

**7. Limitaciones y trabajo futuro.** COFINFAD es un conjunto sintético: las cohortes de edad, la
escalera de `active_products` y la ausencia de señal en las transacciones lo delatan, y las
conclusiones sobre qué determina la fuga no deben trasladarse sin más a clientes reales. La etiqueta es
un puntaje de otro modelo, no la fuga observada, de modo que lo que se aprende es a reproducir ese
puntaje. Con datos reales convendría validar en el tiempo (entrenar con un periodo y probar con el
siguiente), incorporar el coste de cada acción de retención para elegir el umbral, y aprovechar las
secuencias de transacciones con modelos que las traten como series.